# **NIVEL 1**

## Ejercicio 1:

**Exploración inicial del dataset**

Objetivo: entender la estructura general de los datos disponibles y detectar
posibles problemas ANTES de limpiar o transformar nada.

4 fuentes de datos:
  1. sprint10_complex.xlsx   -> dataset principal: encuesta de trabajadores 2025
  2. matriu_distancies.xlsx  -> matriz de distancias (km) entre ciudades
  3. ciutats_context.csv     -> info de contexto por ciudad (coste de vida, transporte, clima)
  4. politica_salarial.json  -> reglas de negocio para calcular incrementos salariales

Por ahora solo exploramos y documentamos problemas. No se corrige nada todavía.

**PASO 1:** Importar datos de Google Drive

In [59]:
from google.colab import drive
drive.mount('/content/drive')
RUTA_DATOS = "/content/drive/MyDrive/Colab Notebooks/Datos_Sprint8/"

import pandas as pd
import numpy as np
import json

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**PASO 2:** Carga del dataset principal y primera exploración de los datos

In [60]:
df_raw = pd.read_excel(RUTA_DATOS + "sprint10_complex.xlsx")
df_raw.head(5)

,Dades de l'enquesta de treballadors 2025,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19
0,Generades aleatòriament per seguretat,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Fills,No Fills,Grup Professional,CÃ rrec,Nombre_fills,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat
3,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 â‚¬",True,NaN,Grup B,Cap de projecte,3.0,True,32108.0,25.0,10.1
4,1,Marc,MuÃ±oz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,H,"2,718 â‚¬",NaN,True,Grup C,Senior analyst,0.0,True,19496.0,10.4,18.7


*Observaciones*

1. El archivo tiene 3 filas de "basura" antes de la cabecera real (un título y una nota "Generades aleatòriament per seguretat"). Pandas interpreta la primera fila como header así que todo sale mal (columnas sin nombre, tipos incorrectos, etc.) Y hay muchos missing values (NaN).

**PASO 3:** Resumen. Para seguir explorando el contenido del primer dataset, indicamos dónde está la cabecera real, para encontrar dónde empieza la tabla. Con distintas funciones de pandas, visualizamos un resumen de los datos.

In [61]:
df = pd.read_excel(RUTA_DATOS + "sprint10_complex.xlsx", header=3)

print("\n--- Dimensiones ---")
print("Filas:", df.shape[0], "| Columnas:", df.shape[1])

print("\n--- Muestra representativa (5 filas aleatorias) ---")
print(df.sample(5, random_state=42))

print("\n--- Tipos de datos ---")
print(df.dtypes)

print("\n--- Filas duplicadas (completas) ---")
print(df.duplicated().sum())

print("\n--- Nulos por columna ---")
print(df.isnull().sum())

print("\n--- Últimas filas (revisar el final del archivo) ---")
print(df.tail(3))

print("\n--- Valores únicos en columnas clave ---")
print("Gènere:\n", df["Gènere"].value_counts(dropna=False))
print("\nGrup Professional:\n", df["Grup Professional"].value_counts(dropna=False))

print("\n--- Muestra de la columna 'Salari mensual' (texto, no numérica) ---")
print(df["Salari mensual"].sample(10, random_state=1).tolist())

print("\n--- DNI duplicados ---")
print(df["DNI"].duplicated().sum())


--- Dimensiones ---
Filas: 1007 | Columnas: 20

--- Muestra representativa (5 filas aleatorias) ---
     Unnamed: 0      Nom       Cognoms        DNI País d'origen     Ciutat  Dia de Naixement  Mes de Naixement  Any de Naixement Gènere Salari mensual Fills No Fills Grup Professional  \
927         927     Anna  PÃ©rez Serra  81260169S       Espanya  Barcelona              11.0               2.0            1983.0      H      1.927 â‚¬  True      NaN            Grup B   
630         630  VÃ­ctor     DomÃ¨nech  22730042T       Espanya   Zaragoza               7.0               3.0            1995.0      H      1,579 â‚¬   NaN     True            Grup B   
682         682   Carmen  DÃ­az Casals  89650979N       Espanya     Murcia              11.0              10.0            1968.0      H      2,425 â‚¬   NaN     True            Grup C   
514         514  VÃ­ctor   Roca MuÃ±oz  25081699D       Espanya      Palma               8.0               2.0            1976.0      D         943â‚¬ 

**PASO 4:** Cargamos y exploramos el dataset de distancias entre ciudades.

In [62]:
# 2. MATRIU_DISTANCIES.XLSX - distancias entre ciudades
print("2. MATRIU_DISTANCIES.XLSX")

df_dist = pd.read_excel(RUTA_DATOS + "matriu_distancies.xlsx")
print("\nDimensiones:", df_dist.shape)
print(df_dist.head())
print("\nTipos de datos:\n", df_dist.dtypes)
print("\nFilas duplicadas:", df_dist.duplicated().sum())
print("\nCiudades incluidas:", list(df_dist.columns[1:])) # ignora 1ra col

2. MATRIU_DISTANCIES.XLSX

Dimensiones: (15, 16)
  Unnamed: 0  Barcelona  Valencia  Sevilla  Zaragoza  Málaga  Murcia  Palma  Las Palmas de Gran Canaria  Bilbao  Alicante  Córdoba  Valladolid   Vigo  Gijón  Hospitalet de Llobregat
0  Barcelona        NaN     303.0    831.0     256.0   770.0   471.0  206.0                      2175.0   469.0     407.0    711.0       576.0  908.0  686.0                      7.0
1   Valencia      303.0       NaN    541.0     246.0   468.0   177.0  260.0                      1874.0   473.0     125.0    421.0       441.0  766.0  632.0                    297.0
2    Sevilla      831.0     541.0      NaN     646.0   158.0   433.0  791.0                      1355.0   703.0     495.0    121.0       486.0  588.0  685.0                    824.0
3   Zaragoza      256.0     246.0    646.0       NaN   628.0   408.0  377.0                      2001.0   246.0     368.0    535.0       320.0  652.0  444.0                    250.0
4     Málaga      770.0     468.0    158.

**PASO 5:** Cargamos y exploramos el dataset de contexto por ciudad.

In [63]:
# 3. CIUTATS_CONTEXT.CSV - contexto por ciudad
print("3. CIUTATS_CONTEXT.CSV")

df_ctx = pd.read_csv(RUTA_DATOS + "ciutats_context.csv", encoding="latin-1")

print("\nDimensiones:", df_ctx.shape)
print(df_ctx.head())
print("\nTipos de datos:\n", df_ctx.dtypes)
print("\nFilas duplicadas:", df_ctx.duplicated().sum())
print("\nCiudades duplicadas (mismo nombre):", df_ctx["Ciutat"].duplicated().sum())
print(df_ctx[df_ctx["Ciutat"].duplicated(keep=False)])

3. CIUTATS_CONTEXT.CSV

Dimensiones: (36, 5)
      Ciutat     País  Cost_vida_index  Transport_public_score  Clima_tipus
0  Barcelona  Espanya               78                     8.5  Mediterrani
1     Madrid  Espanya               75                     8.0  Continental
2   València  Espanya               70                     7.5  Mediterrani
3    Sevilla  Espanya               65                     7.0  Mediterrani
4     Bilbao  Espanya               72                     7.8      Oceànic

Tipos de datos:
 Ciutat                     object
País                       object
Cost_vida_index             int64
Transport_public_score    float64
Clima_tipus                object
dtype: object

Filas duplicadas: 0

Ciudades duplicadas (mismo nombre): 1
   Ciutat          País  Cost_vida_index  Transport_public_score  Clima_tipus
14  Paris        França               85                     9.0      Oceànic
34  Paris  Estats Units               88                     6.0  Continental


**PASO 6:** Cargamos y exploramos el dataset de política salarial.

In [64]:
# 4. POLITICA_SALARIAL.JSON
print("4. POLITICA_SALARIAL.JSON")

with open(RUTA_DATOS + "politica_salarial.json", encoding="utf-8") as f:
    politica = json.load(f) # No es tabular, es un diccionario de reglas

print("\nClaves de primer nivel:", list(politica.keys()))
print("\nContenido completo:")
print(json.dumps(politica, indent=2, ensure_ascii=False))

4. POLITICA_SALARIAL.JSON

Claves de primer nivel: ['base_increment_by_group', 'children_adjustment', 'age_adjustment', 'cost_of_living_adjustment', 'transport_adjustment', 'rules']

Contenido completo:
{
  "base_increment_by_group": {
    "Grup A": 5.0,
    "Grup B": 3.5,
    "Grup C": 2.0,
    "Grup D": 8.0
  },
  "children_adjustment": {
    "has_children": 1.5,
    "no_children": 0.0
  },
  "age_adjustment": {
    "18-29": 0.0,
    "30-44": 1.0,
    "45-59": 2.0,
    "60+": 1.5
  },
  "cost_of_living_adjustment": {
    "low": 0.0,
    "medium": 1.0,
    "high": 2.0
  },
  "transport_adjustment": {
    "low": 0.5,
    "medium": 0.0,
    "high": -0.5
  },
  "rules": {
    "max_total_increment_percent": 15,
    "min_salary_threshold": 800
  }
}


## Ejercicio 2:

**Exploración inicial del dataset**

Validaciones básicas de los datos y decisiones de eliminación.

**Objetivo:** asegurar la calidad mínima de los datos y decidir qué registros
pueden seguir en el análisis y cuáles deben descartarse.

PLAN DE ACCIÓN:
  1. Cargar el dataset con la cabecera correcta
  2. Corregir problemas de codificación (columnas y valores de texto)
  3. Eliminar filas duplicadas
  4. Eliminar filas con datos irrecuperables (vacías, valores imposibles, etc.)
  5. Analizar y tratar DNIs duplicados
  6. Revisar campos obligatorios con nulos

In [65]:
n_inicial = len(df)
print(f"Registros iniciales: {n_inicial}")

Registros iniciales: 1007


**PASO 1:** Corrección de codificación
- El archivo xlsx fue guardado con codificación incorrecta. Los caracteres acentuados aparecen como secuencias del tipo:  'MuÃ±oz','ValÃ¨ncia', 'Ã‰lise', etc.
- Esto pudo ser porque el texto fue codificado en UTF-8 pero leído como Latin-1 (o cp1252), dejando las secuencias de bytes corruptas visibles.
- Solución: invertir el proceso (encode a decode) + mapa manual para los casos donde el segundo byte UTF-8 fue reemplazado por un espacio.


In [66]:
MANUAL_FIXES = {
    "AdriÃ ":  "Adrià",
    "ItÃ lia": "Itàlia",
    "MÃ laga": "Málaga",
    "Ãlvarez": "Álvarez",
}

def fix_mojibake(valor):
    if not isinstance(valor, str):
        return valor
    # reemplazos manuales
    for malo, bueno in MANUAL_FIXES.items():
        valor = valor.replace(malo, bueno)
    # intentar cp1252 a utf-8
    try:
        return valor.encode("cp1252").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        pass
    # intentar latin-1 a utf-8
    try:
        return valor.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return valor

#fnc a columnas
df.columns = [fix_mojibake(c) for c in df.columns]

#fnc a registros
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(fix_mojibake)

print("\n Codificación corregida.")
print("  Columnas tras corrección:", list(df.columns))

print("\n  Verificación: Muestra de valores corregidos:")
for col in ["Nom", "Cognoms", "País d'origen", "Ciutat", "Càrrec"]:
    sample = df[col].dropna().unique()[:4]
    print(f"    {col}: {list(sample)}")


 Codificación corregida.
  Columnas tras corrección: ['Unnamed: 0', 'Nom', 'Cognoms', 'DNI', "País d'origen", 'Ciutat', 'Dia de Naixement', 'Mes de Naixement', 'Any de Naixement', 'Gènere', 'Salari mensual', 'Fills', 'No Fills', 'Grup Professional', 'Càrrec', 'Nombre_fills', 'Te_cotxe', 'Km_anuals', 'Consum_mitja_L_100km', 'Temperatura_mitjana_ciutat']

  Verificación: Muestra de valores corregidos:
    Nom: ['Joana', 'Marc', 'Noa', 'Pol']
    Cognoms: ['Gil Navarro', 'Muñoz', 'Serra', 'Gil']
    País d'origen: ['Espanya', 'Noruega', 'Alemanya', 'Itàlia']
    Ciutat: ['Valladolid', 'Alacant', 'Sevilla', 'Bilbao']
    Càrrec: ['Cap de projecte', 'Senior analyst', 'Tècnic IT', 'Data Analyst']


**PASO 2:** Filas duplicadas completamente
- Una fila exactamente igual a otra no aporta información adicional. Mantenemos la primera ocurrencia y eliminamos las copias.


In [67]:
n_dup = df.duplicated().sum()
print(f"\nFilas duplicadas encontradas: {n_dup}")

df = df.drop_duplicates()
print(f"Eliminadas {n_dup} filas duplicadas exactas. Quedan: {len(df)}")


Filas duplicadas encontradas: 0
Eliminadas 0 filas duplicadas exactas. Quedan: 1007


**PASO 3:** Filas con datos irrecuperables
- En caso de alguna fila completamente vacía, decido eliminarla. No hay forma de saber a qué persona podría corresponder.
- Hay 38 filas con "any de naixement" igual a 3000 o 2026, o sea, fechas imposibles. La decisión es eliminarlas porque no hay modo de saber el año real sin información externa y el año de nacimiento se necesita para calcular otros datos.


In [68]:
cols_datos = [c for c in df.columns if c != "Unnamed: 0"] # excluye col índice
mask_vacia = df[cols_datos].isnull().all(axis=1)
n_vacias = mask_vacia.sum()
df = df[~mask_vacia] # filas que sí tienen data
print(f"\nEliminadas {n_vacias} filas completamente vacías. Quedan: {len(df)}")


mask_any_imposible = df["Any de Naixement"] > 2025
n_any = mask_any_imposible.sum()
df = df[~mask_any_imposible]
print(f"Eliminadas {n_any} filas con año de nacimiento imposible (>2025). Quedan: {len(df)}")


Eliminadas 1 filas completamente vacías. Quedan: 1006
Eliminadas 38 filas con año de nacimiento imposible (>2025). Quedan: 968


**PASO 4:** DNIs duplicados

- Primero busco los DNI duplicados.
- Luego evaluamos si los duplicados comparten info esencial como: nombre, apellido, país, ciudad, género y año de nacimiento.
- Normalizamos Gènere ("H"→"Home", "D"→"Dona") antes de contar coincidencias, porque si no, dos filas de la misma persona podían parecer "distintas" solo por cómo se escribió el género.
- Si sí comparten estos datos, asumo que se trata de la misma persona y elimino el restante de duplicados.
- Si no comparten estos otros datos, me quedo con los que tengan país = "Espanya" porque solo en este país se emite el DNI.
- NOTA: Para los casos restantes de duplicados con país España, como no puedo saber cuál de todos es la persona real, preguntaría a algún stakeholder qué hacer en este caso o buscaría en una base de datos pública a quién pertenece el DNI.

In [69]:
MAPA_GENERE = {"H": "Home", "D": "Dona", "Home": "Home", "Dona": "Dona"}

generos_normalizados = []
for valor in df["Gènere"]:
    if pd.isna(valor):
        generos_normalizados.append(valor)
    else:
        valor_limpio = valor.strip()
        if valor_limpio in MAPA_GENERE:
            generos_normalizados.append(MAPA_GENERE[valor_limpio])
        else:
            generos_normalizados.append(valor_limpio)

df["_Genere_norm"] = generos_normalizados

CAMPOS_CLAVE = ["Nom", "Cognoms", "País d'origen", "Ciutat", "_Genere_norm", "Any de Naixement"]

def campos_coinciden(f1, f2, campos):
    contador = 0
    for c in campos:
        valor1 = f1[c]
        valor2 = f2[c]
        if pd.notna(valor1) and pd.notna(valor2) and valor1 == valor2:
            contador += 1
    return contador

def agrupar_por_persona(grupo, campos_clave):
    idxs = grupo.index.tolist()
    grupos_personas = []

    for idx in idxs:
        fila = grupo.loc[idx]
        encontro_grupo = False

        for grupo_persona in grupos_personas:
            for idx_existente in grupo_persona:
                fila_existente = grupo.loc[idx_existente]
                if campos_coinciden(fila, fila_existente, campos_clave) >= 3:
                    grupo_persona.append(idx)
                    encontro_grupo = True
                    break
            if encontro_grupo:
                break

        if not encontro_grupo:
            grupos_personas.append([idx])

    return grupos_personas

mask_dni_dup = df["DNI"].duplicated(keep=False)
df_dup = df[mask_dni_dup]
print(f"Filas implicadas en DNIs duplicados: {len(df_dup)} ({df_dup['DNI'].nunique()} DNIs distintos)")

filas_copias_coherentes = []           # misma persona repetida - se elimina
filas_contradictorias_resueltas = []   # personas distintas, resuelto por país=Espanya - se elimina
filas_dudosas = []                     # personas distintas, sin forma de decidir - NO se elimina, se marca

for dni, grupo in df_dup.groupby("DNI"):
    grupos_personas = agrupar_por_persona(grupo, CAMPOS_CLAVE)
    if len(grupos_personas) == 1:
        miembros = grupos_personas[0]
        filas_copias_coherentes.extend(miembros[1:])
    else:
        representantes = []
        for miembros in grupos_personas:
            filas_copias_coherentes.extend(miembros[1:])
            representantes.append(miembros[0])
        filas_espana = [r for r in representantes if df.loc[r, "País d'origen"] == "Espanya"]
        if len(filas_espana) == 1:
            for r in representantes:
                if r not in filas_espana:
                    filas_contradictorias_resueltas.append(r)
        else:
            filas_dudosas.extend(representantes)

n_grupo_a_eliminados = len(filas_copias_coherentes)
n_grupo_b_eliminados = len(filas_contradictorias_resueltas)
n_dni_dudosos = len(filas_dudosas)

df["DNI_dudoso"] = df.index.isin(filas_dudosas)
df = df.drop(index=filas_copias_coherentes + filas_contradictorias_resueltas)
df = df.drop(columns="_Genere_norm")

print(f"Filas eliminadas por ser la misma persona duplicada: {len(filas_copias_coherentes)}")
print(f"Filas marcadas como DNI_dudoso (revisión manual): {len(filas_dudosas)}")
print(f"Quedan {len(df)} filas tras depurar DNIs duplicados.")

Filas implicadas en DNIs duplicados: 24 (9 DNIs distintos)
Filas eliminadas por ser la misma persona duplicada: 11
Filas marcadas como DNI_dudoso (revisión manual): 6
Quedan 957 filas tras depurar DNIs duplicados.


**PASO 5:** Campos clave con nulos
- Identificar los campos clave que no deberían contener valores nulos, detectar registros con información insuficiente y eliminar únicamente aquellas filas completamente vacías o con información irrecuperable o no sustituible.
- Primero creamos una lista con los campos clave para buscar nulos justo en esas columnas. Hacemos una suma de ellos.
- Luego buscamos y contamos si alguna fila tiene el campo "Grup professional" nulo porque es un valor categórico que no se puede inferir o imputar.
- Imprimimos el número de filas sin grupo profesional, y de esas, mostramos solo las columnas relevantes, no todas.
- Si hay más de 0 filas sin grupo, imprimimos cuántas se eliminaron.


In [70]:
campos_clave = ["DNI", "Nom", "Cognoms", "Any de Naixement",
                "Gènere", "Grup Professional", "Salari mensual", "Ciutat"]
df[campos_clave].isnull().sum()

,0
DNI,0
Nom,0
Cognoms,0
Any de Naixement,0
Gènere,38
Grup Professional,0
Salari mensual,0
Ciutat,0


In [71]:
mask_sin_grup = df["Grup Professional"].isnull()
n_sin_grup = mask_sin_grup.sum()

print(f"\nFilas sin Grup Professional: {n_sin_grup}")
print(df[mask_sin_grup][campos_clave])

if n_sin_grup > 0:
    df = df[~mask_sin_grup]
    print(f"- Eliminadas {n_sin_grup} fila(s): 'Grup Professional' es un campo obligatorio "
          "no imputable (dato categórico, no inferible de otras columnas).")


Filas sin Grup Professional: 0
Empty DataFrame
Columns: [DNI, Nom, Cognoms, Any de Naixement, Gènere, Grup Professional, Salari mensual, Ciutat]
Index: []


**RESUMEN FINAL**


In [72]:
n_final = len(df)
print("RESUMEN DE ELIMINACIONES")
print(f"  Registros iniciales:                      {n_inicial}")
print(f"  - Filas completamente duplicadas:        -{n_dup}")
print(f"  - Filas completamente vacías:            -{n_vacias}")
print(f"  - Año de nacimiento imposible (>2025):   -{n_any}")
print(f"  - DNIs duplicados (copias coherentes):   -{n_grupo_a_eliminados}")
print(f"  - DNIs duplicados (contradicción resuelta por país): -{n_grupo_b_eliminados}")
print(f"  - Sin Grup Professional (irrecuperable): -{n_sin_grup}")
print(f"  ──────────────────────────────────────────")
print(f"  Registros válidos para el análisis:       {n_final}")
print(f"  (de los cuales {n_dni_dudosos} quedan marcados como DNI_dudoso, sin eliminar)")
print()
print("Nulos restantes en campos clave (a tratar en ejercicio 3):")
print(df[campos_clave].isnull().sum())

RESUMEN DE ELIMINACIONES
  Registros iniciales:                      1007
  - Filas completamente duplicadas:        -0
  - Filas completamente vacías:            -1
  - Año de nacimiento imposible (>2025):   -38
  - DNIs duplicados (copias coherentes):   -11
  - DNIs duplicados (contradicción resuelta por país): -0
  - Sin Grup Professional (irrecuperable): -0
  ──────────────────────────────────────────
  Registros válidos para el análisis:       957
  (de los cuales 6 quedan marcados como DNI_dudoso, sin eliminar)

Nulos restantes en campos clave (a tratar en ejercicio 3):
DNI                   0
Nom                   0
Cognoms               0
Any de Naixement      0
Gènere               38
Grup Professional     0
Salari mensual        0
Ciutat                0
dtype: int64


## Ejercicio 3:

**Transformaciones necesarias y preparación del dataset**

**Objetivo:** dejar el dataset coherente, analizable, estructuralmente correcto SIN eliminar registros (solo transformarlos, normalizarlos e imputar).
Partimos del DataFrame del ejercicio 2 (df_clean) con 957 filas.

**PASO 1:** Normalización de valores categóricos (caso Gènere)

- La columna mezcla catalán, castellano y abreviaturas (D-Dona-F, H-Home-M, A para Altres, NC para No consta y ?? un valor inválido, estos últimos dos se tratan como faltantes).



In [73]:
MAPA_GENERE = {
    "D": "Dona", "Dona": "Dona", "F": "Dona",
    "H": "Home", "Home": "Home", "M": "Home",
    "A": "Altres",
    "NC": np.nan,
    "??": np.nan,
}

genere_normalizado = []
for valor in df["Gènere"]:
    if pd.isna(valor):
        genere_normalizado.append(np.nan)
    else:
        valor_limpio = valor.strip()
        genere_normalizado.append(MAPA_GENERE.get(valor_limpio, valor_limpio))

df["Gènere"] = genere_normalizado

print("Gènere tras normalizar:")
print(df["Gènere"].value_counts(dropna=False))

Gènere tras normalizar:
Gènere
Dona      524
Home      348
NaN        61
Altres     24
Name: count, dtype: int64


**PASO 2:** Unificar información de hijos en una sola columna
- Primero me quedo con el número de hijos. Si no existe, y "No Fills" = True, quiere decir que hay 0 hijos. En cualquier otro caso, lo trato como un NaN porque no sabemos si tiene o no.
- Creo una lista que contendrá el número de hijos.
- Recorro las filas del dataframe con un bucle que devuelve los valores de número de hijos, si hay hijos o si no lso hay.
- Si no hay valores perdidos en valor_nombre (número de hijos), se va llenando mi lista de número de hijos.
- En caso de que "no hijos" sea True, e "hijos" sea diferente de True, adjunto un valor 0 a mi lista de número de hijos.
- Para los demás casos, agrego un valor nulo a mi lista.
- Creo la columna "Numero_fills" que contiene mi lista anterior.
- Luego elimino las columnas anteriores que son redundantes.
- Imprimo algunos valores de mi nueva columna junto con sus índices.
- Finalmente muestro algunos registros del dataframe.

In [74]:
numero_fills = []
for i in range(len(df)):
    valor_nombre = df["Nombre_fills"].iloc[i]
    valor_fills = df["Fills"].iloc[i]
    valor_no_fills = df["No Fills"].iloc[i]

    if pd.notna(valor_nombre):
        numero_fills.append(valor_nombre)
    elif valor_no_fills == True and valor_fills != True:
        numero_fills.append(0)
    else:
        numero_fills.append(pd.NA)

df["Numero_fills"] = numero_fills
df = df.drop(columns=["Fills", "No Fills", "Nombre_fills"])

print(df["Numero_fills"].value_counts(dropna=False).sort_index())
df.head()

Numero_fills
0.0     546
1.0     163
2.0      78
3.0      61
4.0      77
<NA>     32
Name: count, dtype: int64


,Unnamed: 0,Nom,Cognoms,DNI,País d'origen,Ciutat,Dia de Naixement,Mes de Naixement,Any de Naixement,Gènere,Salari mensual,Grup Professional,Càrrec,Te_cotxe,Km_anuals,Consum_mitja_L_100km,Temperatura_mitjana_ciutat,DNI_dudoso,Numero_fills
0,0,Joana,Gil Navarro,66722344X,Espanya,Valladolid,23.0,3.0,1958.0,Dona,"1,469 €",Grup B,Cap de projecte,True,32108.0,25.0,10.1,False,3.0
1,1,Marc,Muñoz,48840994W,Espanya,Alacant,8.0,11.0,1960.0,Home,"2,718 €",Grup C,Senior analyst,True,19496.0,10.4,18.7,False,0.0
2,2,Noa,Serra,14308421X,Espanya,Alacant,27.0,4.0,1961.0,Dona,1358 euros,Grup A,Tècnic IT,NaN,NaN,NaN,16.7,False,4.0
3,3,Pol,Gil,58586340F,Espanya,Sevilla,13.0,10.0,1985.0,Home,"1,478 €",Grup B,Data Analyst,False,NaN,NaN,18.3,False,2.0
4,4,David,Vila,82070937P,Espanya,Bilbao,29.0,11.0,1965.0,Dona,"1,284 €",Grup B,Administratiu,True,NaN,11.8,13.1,False,0


**PASO 3:** Coherencia coche/ km anuales/ consumo
- Km_anuals debe ser no-negativo (0 es válido porque es congruente con no tener coche). Y de máximo 6 cifras (excluye errores de captura u "outliers").
- Mediana de referencia: km anuales típicos de quien SÍ tiene coche.
- Incoherencia real: consta consumo medio pero no kilometraje anual. No se puede derivar km_anuals a partir del consumo (L/100km mide eficiencia, no uso), así que imputamos con la mediana de quienes tienen coche.

In [75]:
LIMITE_KM = 999_999
mask_km_invalido = df["Km_anuals"].notna() & ((df["Km_anuals"] < 0) | (df["Km_anuals"] > LIMITE_KM))
n_km_invalido = mask_km_invalido.sum()
print(f"Valores de Km_anuals inválidos (negativos o >6 cifras): {n_km_invalido}")
df.loc[mask_km_invalido, "Km_anuals"] = np.nan

mediana_km_con_coche = df.loc[df["Te_cotxe"] == True, "Km_anuals"].median()
print("Mediana de Km_anuals (Te_cotxe=True, ya limpia):", mediana_km_con_coche)

mask_incoherente = df["Km_anuals"].isnull() & df["Consum_mitja_L_100km"].notna()
n_incoherente = mask_incoherente.sum()
print(f"Filas con Consum pero sin Km_anuals (incoherentes, se imputan): {n_incoherente}")
df.loc[mask_incoherente, "Km_anuals"] = mediana_km_con_coche

Valores de Km_anuals inválidos (negativos o >6 cifras): 6
Mediana de Km_anuals (Te_cotxe=True, ya limpia): 60880.5
Filas con Consum pero sin Km_anuals (incoherentes, se imputan): 3


**PASO 4:** Limpieza del salario
- Hay que remover valores extraños en los salarios, así que creo una función "limpiar_salario" que:
  - no hace nada con los nulos,
  - quita espacios al principio y al final,
  - revisa si el contenido tiene sentido o si el salario está escrito con palabras y lo cambia a número con un diccionario de referencia,
  - quita los símbolos de euros, la palabra euros, comas, etc.
  - al final, la función devuelve el valor en float
- Creo una nueva columna ("Salari_num") que contiene lo del salario mensual con la función limpiar_salario aplicada.
- Imprimo la suma de los nulos y la columna nueva.

In [76]:
PALABRAS_NUMERO = {
    "mil vuit-cents": 1800,
    "dos mil": 2000,
    "tres mil": 3000,
}
PALABRAS_SIN_DATO = {"molts", "n/d"}

def limpiar_salario(valor):
    if pd.isna(valor):
        return None
    v = valor.strip()
    if v in PALABRAS_SIN_DATO:
        return None
    if v in PALABRAS_NUMERO:
        return PALABRAS_NUMERO[v]

    v = v.replace("â‚¬", "").replace("€", "").replace("euros", "").replace("euro", "")
    v = v.replace(",", "").replace(".", "")
    v = v.strip()
    return float(v)

df["Salari_num"] = df["Salari mensual"].apply(limpiar_salario)
df = df.drop(columns=["Salari mensual"])

print("Nulos en Salari_num:", df["Salari_num"].isnull().sum())
print(df["Salari_num"].head())

Nulos en Salari_num: 13
0    1469.0
1    2718.0
2    1358.0
3    1478.0
4    1284.0
Name: Salari_num, dtype: float64


**PASO 5:** Imputación de salario
- Para los casos que no tienen "Salari_num", me baso en el criterio siguiente: promedio de salario dentro del mismo "Grup Professional".
- Calculamos la media de "Salari_num" de cada grupo profesional, ".transform()" devuelve una serie del mismo tamaño que el DataFrame original, con un valor por cada fila (el promedio del grupo al que pertenece esa fila específica).
- Guardamos en mask_falta_salario todos los que tienen nulos y luego hacemos un conteo de ellos.
- df.loc[mask_falta_salario, "Salari_num"] selecciona, dentro de df, únicamente las filas donde falta el salario (mask_falta_salario es True), y específicamente la columna "Salari_num" de esas filas.
- promedio_por_grupo[mask_falta_salario] selecciona, de esa serie de promedios que ya tiene un valor por fila, solo los valores correspondientes a esas mismas filas con salario faltante.
- Imprimimos cuántas veces se imputaron los promedios, y la suma de los nulos que hay ahora.

In [77]:
promedio_por_grupo = df.groupby("Grup Professional")["Salari_num"].transform("mean")

mask_falta_salario = df["Salari_num"].isnull()
n_falta_salario = mask_falta_salario.sum()
df.loc[mask_falta_salario, "Salari_num"] = promedio_por_grupo[mask_falta_salario]

print(f"Salarios imputados por Grup Professional: {n_falta_salario}")
print(f"Nulos restantes en Salari_num: {df['Salari_num'].isnull().sum()}")
df["Salari_num"].head(10)

Salarios imputados por Grup Professional: 13
Nulos restantes en Salari_num: 0


,Salari_num
0,1469.0
1,2718.0
2,1358.0
3,1478.0
4,1284.0
5,2069.0
6,2349.0
7,2855.0
8,1338.0
9,1450.0


**PASO 6:** Validar la temperatura media
- Hay algunos valores de temperatura por ciudad que no hacen sentido (-10 grados en Alicante o Las Palmas, por ejemplo). Así que los excluimos como "valores centinela".
- Imputamos con el promedio de temperatura de la MISMA ciudad (las 44 ciudades
tienen al menos un dato válido, no hace falta subir a un nivel superior como país).

In [78]:
VALORES_CENTINELA = [-10.0, -5.0, 38.0, 42.0]
mask_temp_invalida = df["Temperatura_mitjana_ciutat"].isin(VALORES_CENTINELA)
n_temp_invalida = mask_temp_invalida.sum()
df.loc[mask_temp_invalida, "Temperatura_mitjana_ciutat"] = np.nan

promedio_por_ciutat = df.groupby("Ciutat")["Temperatura_mitjana_ciutat"].transform("mean")
mask_falta_temp = df["Temperatura_mitjana_ciutat"].isnull()
n_falta_temp = mask_falta_temp.sum()
df.loc[mask_falta_temp, "Temperatura_mitjana_ciutat"] = promedio_por_ciutat[mask_falta_temp]

print(f"Valores centinela detectados y anulados: {n_temp_invalida}")
print(f"Total imputado por promedio de ciudad: {n_falta_temp}")
print(f"Nulos restantes: {df['Temperatura_mitjana_ciutat'].isnull().sum()}")
df['Temperatura_mitjana_ciutat'].head(10)

Valores centinela detectados y anulados: 62
Total imputado por promedio de ciudad: 126
Nulos restantes: 0


,Temperatura_mitjana_ciutat
0,10.1
1,18.7
2,16.7
3,18.3
4,13.1
5,19.9
6,17.8
7,15.4
8,7.6
9,20.1


# Ejercicio 4

**Primeros análisis descriptivos**

**Objetivo:** obtener una primera visión global del dataset una vez aplicadas las transformaciones básicas, mediante estadísticas descriptivas y comparaciones simples entre grupos relevantes.

**PASO 1:**
- Tabla resumen 1: salario por género
- Selecciono la columna "Salari_num" agrupándola por género y calculamos media, mediana, mínimo y máximo.
- De esa tabla resultante, ordenamos las filas por promedio de mayor a menor.
- Dejamos solo dos decimales para cada valor e imprimimos la tabla nueva.

In [79]:
# --- Tabla resumen: salario por género ---
tabla_salario_genere = df.groupby("Gènere")["Salari_num"].agg(["mean", "median", "min", "max"])
tabla_salario_genere = tabla_salario_genere.sort_values("mean", ascending=False)
tabla_salario_genere = tabla_salario_genere.round(2)

print(tabla_salario_genere)

           mean  median    min     max
Gènere                                
Altres  1699.27  1205.5  673.0  3600.0
Home    1693.62  1507.5  726.0  3672.0
Dona    1477.02  1341.5  701.0  3564.0


**PASO 2:**
- Tabla resumen 2: salario medio por género (filas) y país de origen (columnas)
- Creo una pivot table que contiene una columna extra con el promedio total por género (columna "Media") y el promedio total por país (fila "Media").
- El color más intenso indica el valor más alto.

In [80]:
tabla_pivote = pd.pivot_table(
    df,
    values="Salari_num",
    index="Gènere",
    columns="País d'origen",
    aggfunc="mean",
    margins=True,          # añade fila y columna de "Media"
    margins_name="Media"
).round(2)

# Formato condicional: color más intenso = valor más alto
tabla_pivote_formateada = tabla_pivote.style.background_gradient(cmap="YlOrRd", axis=None)
tabla_pivote_formateada

País d'origen,Alemanya,Espanya,França,Itàlia,Noruega,Media
Gènere,,,,,,
Altres,nan,1784.270000,1104.330000,nan,nan,1699.270000
Dona,1418.170000,1466.580000,1414.260000,1498.060000,1779.750000,1477.020000
Home,2140.200000,1695.480000,1654.340000,1676.610000,1510.760000,1693.620000
Media,1598.680000,1567.220000,1496.700000,1567.930000,1668.220000,1567.100000
